In [0]:
dbutils.widgets.text("secret_scope", "eventhub")
dbutils.widgets.text("secret_key", "eh-connection-string")
dbutils.widgets.text("eh_namespace", "evhua5816bd")
dbutils.widgets.text("eh_name", "roksolana-wikipedia-recentchange")
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_landing", "roksolana_shendiu770")
dbutils.widgets.text("schema_bronze", "roksolana_shendiu770_bronze")
dbutils.widgets.text("target_table_name", "wikipedia_recentchange_bronze")
dbutils.widgets.text("checkpoint_subdir", "wikipedia_recentchange")
dbutils.widgets.text("starting_offsets", "earliest")
dbutils.widgets.text("trigger_seconds", "30")
dbutils.widgets.text("max_offsets_per_trigger", "50000")
dbutils.widgets.text("fail_on_data_loss", "false")
dbutils.widgets.text("kafka_request_timeout_ms", "60000")
dbutils.widgets.text("kafka_session_timeout_ms", "30000")
dbutils.widgets.text("run_duration_seconds", "300")

SECRET_SCOPE = dbutils.widgets.get("secret_scope")
SECRET_KEY = dbutils.widgets.get("secret_key")
EH_NAMESPACE = dbutils.widgets.get("eh_namespace")
EH_NAME = dbutils.widgets.get("eh_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA_LANDING = dbutils.widgets.get("schema_landing")
SCHEMA_BRONZE = dbutils.widgets.get("schema_bronze")
TARGET_TABLE_NAME = dbutils.widgets.get("target_table_name")
CHECKPOINT_SUBDIR = dbutils.widgets.get("checkpoint_subdir")
STARTING_OFFSETS = dbutils.widgets.get("starting_offsets")
TRIGGER_SECONDS = int(dbutils.widgets.get("trigger_seconds"))
MAX_OFFSETS_PER_TRIGGER = dbutils.widgets.get("max_offsets_per_trigger")
FAIL_ON_DATA_LOSS = dbutils.widgets.get("fail_on_data_loss")
KAFKA_REQUEST_TIMEOUT_MS = dbutils.widgets.get("kafka_request_timeout_ms")
KAFKA_SESSION_TIMEOUT_MS = dbutils.widgets.get("kafka_session_timeout_ms")
RUN_DURATION_SECONDS = int(dbutils.widgets.get("run_duration_seconds"))

required_params = {
    "eh_namespace": EH_NAMESPACE,
    "catalog": CATALOG,
    "schema_landing": SCHEMA_LANDING,
    "schema_bronze": SCHEMA_BRONZE,
}
missing = [name for name, value in required_params.items() if not value]
if missing:
    raise ValueError(f"Missing required parameters: {', '.join(missing)}")

CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_checkpoints/{CHECKPOINT_SUBDIR}"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.{TARGET_TABLE_NAME}"

EH_CONN_STR = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": f"{EH_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": EH_NAME,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="$ConnectionString" password="{EH_CONN_STR}";'
    ),
    "kafka.request.timeout.ms": KAFKA_REQUEST_TIMEOUT_MS,
    "kafka.session.timeout.ms": KAFKA_SESSION_TIMEOUT_MS,
    "maxOffsetsPerTrigger": MAX_OFFSETS_PER_TRIGGER,
    "failOnDataLoss": FAIL_ON_DATA_LOSS,
    "startingOffsets": STARTING_OFFSETS,
}

In [0]:
import time
from pyspark.sql.functions import col, from_json, current_timestamp, expr

payload_ddl = """
    id BIGINT,
    type STRING,
    title STRING,
    user STRING,
    bot BOOLEAN,
    minor BOOLEAN,
    timestamp BIGINT,
    wiki STRING,
    server_name STRING,
    length STRUCT<old: BIGINT, new: BIGINT>
"""

raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

parsed_stream = (
    raw_stream
    .withColumn("body_str", col("value").cast("string"))
    .withColumn("payload", from_json(col("body_str"), payload_ddl))
    .select(
        col("payload.id").alias("wiki_event_id"),
        col("payload.type").alias("event_type"),
        col("payload.title"),
        col("payload.user"),
        col("payload.bot"),
        col("payload.minor"),
        col("payload.timestamp").alias("wiki_event_timestamp"),
        col("payload.wiki"),
        col("payload.server_name"),
        col("payload.length"),
        col("topic").alias("eh_name"),
        col("partition").alias("eh_partition"),
        col("offset").alias("eh_offset"),
        col("timestamp").alias("eh_enqueued_timestamp"),
        current_timestamp().alias("etl_processed_timestamp"),
        expr("uuid()").alias("etl_rec_uuid"),
    )
)

query = (
    parsed_stream.writeStream
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(processingTime=f"{TRIGGER_SECONDS} seconds")
    .toTable(TARGET_TABLE)
)

time.sleep(RUN_DURATION_SECONDS)
query.stop()